# Practice
## Cleaning and Decomposing Hourly Electricity Demand

### Scenario

You are working as a junior data analyst for an electricity utility.

You receive one year of **hourly electricity-demand data** measured in megawatts (MW).

The raw dataset may contain common time-series data-quality issues such as:

- timestamps stored in an unsuitable data type;
- observations that are not arranged chronologically;
- duplicated timestamps;
- missing demand observations.

Your task is to prepare the data and investigate its time-series structure.

---

### Main decomposition idea

For an additive decomposition:

$$
Y_t = T_t + S_t + R_t
$$

where:

- $Y_t$ = observed electricity demand;
- $T_t$ = trend;
- $S_t$ = seasonality;
- $R_t$ = residual.


## Learning objectives

By the end of this activity, you should be able to:

1. load and inspect hourly time-series data;
2. prepare a timestamp column;
3. check chronological ordering;
4. identify and handle duplicated timestamps;
5. identify and handle missing observations;
6. identify the observation frequency;
7. inspect and plot the observed time series;
8. investigate the time-of-day pattern;
9. decompose the series into trend, seasonality, and residual;
10. explain the main insight from each graph.


# Part A — Load and Inspect the Raw Data

## Task 1 — Import the required libraries

Import the libraries needed for:

- data handling;
- plotting;
- time-series decomposition.



In [12]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

pd.set_option("display.max_columns", None)


## Task 2 — Load the dataset

Load the provided CSV file into a DataFrame and display several rows.


In [13]:
df_raw = pd.read_csv("cambodia_hourly_electricity_demand_practice.csv")
df_raw

,timestamp,electricity_demand_mw
0,2025-02-21 02:00:00,2007.5
1,2025-12-04 21:00:00,3195.9
2,2025-03-06 23:00:00,2802.9
3,2025-05-31 21:00:00,3615.9
4,2025-12-07 15:00:00,2887.6
...,...,...
8757,2025-08-27 22:00:00,3156.6
8758,2025-08-05 07:00:00,2436.8
8759,2025-08-13 14:00:00,3051.3
8760,2025-02-05 20:00:00,3373.3


### Record your observations

1. What are the columns? 
 - there are 2 columns 
2. Which column represents time?
 - the first columns represent time
3. Which column represents electricity demand?
 - the second row represent electricity demand
4. What does one row represent?
  - each row represent data time and electricity demand values
5. Do the timestamps appear to be ordered?
   - the timestamps isn't appear to be in ordered 


# Part B — Prepare the Time Column

## Task 3 — Inspect the DataFrame structure

Inspect:

- column data types;
- number of non-missing values;
- overall DataFrame structure.

**Hint — useful functions/attributes:**  
`dtypes` · `info()`


In [14]:
df_raw.dtypes

timestamp                 object
electricity_demand_mw    float64
dtype: object

In [15]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8762 entries, 0 to 8761
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   timestamp              8762 non-null   object 
 1   electricity_demand_mw  8753 non-null   float64
dtypes: float64(1), object(1)
memory usage: 137.0+ KB


### Interpretation

Write your answers:

- Current timestamp data type: Object 
- Electricity-demand data type: Float62
- Number of rows: 8,762
- Any missing values visible from the structure?
  - There are 9 missing values from the structure


## Task 4 — Convert the timestamp column to datetime

Convert the time column into an appropriate datetime representation.

Then check the data type again.

**Hint — useful functions:**  
`to_datetime()`


In [16]:
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], format='%Y-%m-%d %H:%M:%S')

# Part C — Check Chronological Order

## Task 5 — Check whether observations are ordered in time

Determine whether timestamps are already arranged from earliest to latest.

**Hint — useful attribute:**  
`is_monotonic_increasing`


In [17]:
df_raw['timestamp'].is_monotonic_increasing

False

### Question

Why could an unsorted time column produce a misleading line graph?


## Task 6 — Sort the observations

Sort the DataFrame chronologically.

Then verify the ordering again.

**Hint — useful functions:**  
`sort_values()` · `reset_index()`


In [18]:
df_raw.sort_values(by='timestamp', inplace=True)
df_raw.reset_index()

,index,timestamp,electricity_demand_mw
0,1022,2025-01-01 00:00:00,2188.3
1,6781,2025-01-01 01:00:00,2053.2
2,7085,2025-01-01 02:00:00,1931.1
3,7865,2025-01-01 03:00:00,1994.7
4,8428,2025-01-01 04:00:00,1842.2
...,...,...,...
8757,158,2025-12-31 19:00:00,3419.9
8758,2219,2025-12-31 20:00:00,3558.4
8759,730,2025-12-31 21:00:00,3372.7
8760,712,2025-12-31 22:00:00,3143.9


# Part D — Check Duplicate Timestamps

## Task 7 — Identify duplicated timestamps

Find all rows whose timestamp appears more than once.

**Hint — useful functions:**  
`duplicated()`


In [19]:
df_raw['timestamp'].duplicated().sum()

np.int64(2)

### Interpretation

For each duplicate you find, think about:

- Could it be a repeated system export?
  - : repeated system export can be happened  due network retries (failed export mid-way) and manual override.
- Could it be a repeated measurement?
  - :  repeated measurement can be also happend due to merging error between old and new data.
- Could both records be valid?
  - : both records can be valid if we surveyed a user at exact same datetime and receiving the same inputs.
- Should duplicates be removed, summed, averaged, or investigated?
  - : at first we should investigated first whenether the input is come from the same user or not if it from different user we will perfrom group by /sum/ mean and will remove if we found out the input comming from the same user.

For this exercise, choose a reasonable approach and briefly justify it.


## Task 8 — Handle duplicated timestamps

Apply your chosen approach so that each timestamp appears only once.

Then verify the result.

**Hint — useful functions:**  
`groupby()` · `mean()` · `sum()` · `duplicated()`


In [20]:
df_raw = df_raw.groupby(df_raw.index).mean()

In [21]:
df_raw.duplicated().sum()

np.int64(0)

# Part E — Check Missing Values

## Task 9 — Count missing values

Check how many missing values exist in each column.

**Hint — useful functions:**  
`isna()` · `sum()`


In [23]:
df_raw.isna().sum()

timestamp                0
electricity_demand_mw    9
dtype: int64

## Task 10 — Display the missing observations

Show the rows where the electricity-demand measurement is missing.

**Hint — useful functions:**  
`isna()` · `loc`


In [24]:
df_raw.isna().loc[df_raw.isna().any(axis=1)]

,timestamp,electricity_demand_mw
1908,False,True
2629,False,True
5218,False,True
5599,False,True
6690,False,True
6878,False,True
8117,False,True
8284,False,True
8341,False,True


### Interpretation

Answer:

1. Does a missing electricity-demand value mean demand was zero?
 - : No, it doesn't it contain 9 missing Values.
2. What are some possible real-world reasons for a missing hourly measurement?
 - : in  a real world this can be system error from provider or  consumer failed to provided their daily usage data.


# Part F — Datetime Index and Interpolation

## Task 11 — Set the timestamp as the DataFrame index

Convert the timestamp column into the time index of the DataFrame.

**Hint — useful functions:**  
`set_index()`


In [25]:
df_raw.set_index('timestamp', inplace=True)

## Task 12 — Fill the missing demand observations

Use time-based interpolation to estimate the small gaps.

Afterward, confirm whether any missing values remain.

**Hint — useful functions:**  
`interpolate()` · `isna()` · `sum()`


In [28]:

df_raw.interpolate(method='time', inplace=True)
df_raw['electricity_demand_mw'].isna().sum()

np.int64(0)

### Reflection

Explain in your own words:

> Why is an interpolated value an estimate rather than a true observed value?


# Part G — Check the Time Structure

## Task 13 — Inspect the time range and frequency

Determine:

- first timestamp;
- last timestamp;
- number of observations;
- inferred observation frequency.

**Hint — useful functions/attributes:**  
`min()` · `max()` · `len()` · `infer_freq()`


In [29]:
df_raw['timestamp'].min()

KeyError: 'timestamp'

### Interpretation

1. What is the observation frequency?
2. How many observations correspond to one day?
3. How many observations correspond to one week?


# Part H — Descriptive Statistics

## Task 14 — Summarize electricity demand

Generate descriptive statistics for the electricity-demand variable.

**Hint — useful functions:**  
`describe()`


### Record the values

- Count:
- Mean:
- Standard deviation:
- Minimum:
- 25th percentile:
- Median:
- 75th percentile:
- Maximum:

### Interpretation question

What can these statistics tell you, and what important time-related information do they **not** tell you?


# Part I — Plot the Observed Time Series

## Task 15 — Plot the complete time series

Create a line graph showing electricity demand over the entire available period.

Include:

- a meaningful title;
- x-axis label;
- y-axis label.

**Hint — useful functions:**  
`figure()` · `plot()` · `title()` · `xlabel()` · `ylabel()` · `show()`


### Graph interpretation

Write at least **three observations** from the graph.

Consider:

- overall movement;
- repeated patterns;
- unusually high or low values;
- whether the graph is too dense to inspect short-term behavior.


# Part J — Zoom In to Inspect the Daily Pattern

## Task 16 — Plot a shorter time window

Choose a short period that is long enough to show several complete daily cycles.

Create a line graph for that period.

**Hint — useful functions/methods:**  
`loc` · `figure()` · `plot()` · `show()`


### Graph interpretation

Look for:

- low-demand hours;
- morning increase;
- daytime behavior;
- evening behavior;
- differences between weekdays and weekends.

Write your observations without assuming that every day must look identical.


# Part K — Average Demand by Hour of Day

## Task 17 — Calculate average demand for each hour

Group all observations by **hour of day** and calculate the average electricity demand.

This is the one small extension from the earlier lesson, but it reuses the same grouping-and-average idea.

**Hint — useful functions/attributes:**  
`groupby()` · `mean()` · `index.hour`


## Task 18 — Plot average demand by hour of day

Create a line graph of the 24 hourly averages.

**Hint — useful functions:**  
`figure()` · `plot()` · `xticks()` · `title()` · `xlabel()` · `ylabel()` · `show()`


### Graph interpretation

Answer:

1. Around what time is average demand lowest?
2. When does demand begin increasing?
3. Is there a daytime peak?
4. Is there an evening peak?
5. Which period appears to have the highest average electricity usage?
6. What real-life activities might explain this pattern?


# Part L — Decompose the Time Series

## Task 19 — Choose an appropriate seasonal period

Before running decomposition, determine the seasonal period that represents one **daily cycle**.

Think about:

- the observation frequency;
- the number of observations in one day.

Write your chosen value and justify it.

**Hint — useful functions/classes:**  
`STL()` · `fit()`


## Task 20 — Save the decomposition components

Create separate columns for:

- trend;
- seasonality;
- residual.

**Hint — useful attributes:**  
`trend` · `seasonal` · `resid`


# Part M — Trend

## Task 21 — Plot the trend component

Create a line graph showing only the estimated trend.

**Hint — useful functions:**  
`figure()` · `plot()` · `title()` · `xlabel()` · `ylabel()` · `show()`


### Graph interpretation

Explain:

1. Is the underlying demand level increasing, decreasing, or changing in another way?
2. Is the trend smoother than the observed series?
3. What does the trend tell you that the raw hourly graph did not show clearly?


# Part N — Seasonality

## Task 22 — Plot the seasonal component

Choose a short period that makes the repeated daily seasonal pattern easy to inspect.

**Hint — useful functions/methods:**  
`loc` · `figure()` · `plot()` · `axhline()` · `show()`


### Graph interpretation

Answer:

1. Does the pattern repeat?
2. How often does it repeat?
3. During which hours is the seasonal effect generally positive?
4. During which hours is it generally negative?
5. What does a positive seasonal value mean?
6. What does a negative seasonal value mean?


# Part O — Residual

## Task 23 — Plot the residual component

Create a residual line graph and include a horizontal reference line at zero.

**Hint — useful functions:**  
`figure()` · `plot()` · `axhline()` · `show()`


### Graph interpretation

Answer:

1. Do residuals fluctuate around zero?
2. Do they show the same obvious daily repetition as the seasonal component?
3. Are there unusually large residuals?
4. What real-world situations could produce a large residual?


# Part P — Compare All Components

## Task 24 — Plot all components together

Create four vertically aligned plots:

1. observed demand;
2. trend;
3. seasonality;
4. residual.

**Hint — useful functions:**  
`subplots()` · `plot()` · `set_title()` · `axhline()` · `tight_layout()` · `show()`


### Interpretation

Use the relationship:

$$
Y_t = T_t + S_t + R_t
$$

Explain how the four graphs tell one connected story.

Write at least one sentence for each:

- Observed:
- Trend:
- Seasonality:
- Residual:


# Part Q — Verify the Components Numerically

## Task 25 — Build a component table

Create a table containing:

- observed electricity demand;
- trend;
- seasonality;
- residual.

Display a short period containing several hourly observations.

**Hint — useful functions/methods:**  
`loc` · `round()`


### Manual check

Choose one row from your table and verify:

$$
\text{Observed}
\approx
\text{Trend}
+
\text{Seasonality}
+
\text{Residual}
$$

Record your calculation below.


# Part R — Main Achievement Challenge

## Task 26 — Compare selected hours

Using your hourly-average result, compare average electricity demand at:

- 03:00;
- 10:00;
- 20:00.

Then explain which is highest and why that pattern is reasonable.

**Hint — useful functions/methods:**  
`groupby()` · `mean()` · `loc`


# Optional Challenge — Extreme Timestamps

This challenge is optional because it introduces two additional methods that are **not required for the core lesson**.

Try to find:

1. the timestamp with the highest electricity demand;
2. the timestamp with the lowest electricity demand.

**Optional hints:**  
`idxmax()` · `idxmin()`

After finding them, explain whether the times make sense based on the daily demand pattern you discovered.


# Final Takeaway

You started with a messy operational-looking dataset and completed:

$$
\boxed{
\text{Load}
\rightarrow
\text{Convert Time}
\rightarrow
\text{Sort}
\rightarrow
\text{Handle Duplicates}
\rightarrow
\text{Interpolate Missing Values}
}
$$

Then you investigated:

$$
\boxed{
\text{Observed Demand}
=
\text{Trend}
+
\text{Daily Seasonality}
+
\text{Residual}
}
$$

The important achievement is not just that the code ran.

You should now be able to connect the graph to a real operational story:

> **Electricity demand changes according to when people and businesses use electricity.**
 

### Final questions

1. Which step in the cleaning process was most important to you, and why?
2. Which graph gave you the clearest insight into electricity-demand behavior?
3. What did decomposition reveal that the original line graph did not?
4. What would you want to investigate next before building a forecasting model?
